# Classical Approaches


## Import Libraries

In [ ]:
import pandas as pd
import importlib 
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
	sys.path.append(module_path)

import src.utils
importlib.reload(src.utils)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from pprint import pprint

from src.utils import create_one_hot_encoding, get_unique_champs_from_df

## Bag of Champions + Logistic Regression

### Splitting the Dataset


In [2]:
matches = pd.read_csv("../data/TeamMatchTbl.csv")

dropped_cols = [
	"TeamID", "MatchFk", "BlueBaronKills",
	"BlueRiftHeraldKills", "BlueDragonKills",
	"BlueTowerKills", "BlueKills", "RedBaronKills",
	"RedRiftHeraldKills", "RedDragonKills", "RedTowerKills",
	"RedKills", "RedWin"
]

print("Dropped Columns")
pprint(dropped_cols)

train_val_df, test_df = train_test_split(matches, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)

train_df = train_df.drop(columns=dropped_cols)
val_df = val_df.drop(columns=dropped_cols)
test_df = test_df.drop(columns=dropped_cols)

train_df.head()

Dropped Columns
['TeamID',
 'MatchFk',
 'BlueBaronKills',
 'BlueRiftHeraldKills',
 'BlueDragonKills',
 'BlueTowerKills',
 'BlueKills',
 'RedBaronKills',
 'RedRiftHeraldKills',
 'RedDragonKills',
 'RedTowerKills',
 'RedKills',
 'RedWin']


,B1Champ,B2Champ,B3Champ,B4Champ,B5Champ,R1Champ,R2Champ,R3Champ,R4Champ,R5Champ,BlueWin
117776,77,3,887,36,233,106,84,202,85,429,1
105311,83,141,103,901,201,777,28,25,81,518,0
41139,82,234,28,804,26,887,64,136,222,89,0
24402,24,11,13,115,43,36,517,777,202,267,0
118127,904,83,90,901,25,17,266,127,21,350,0


In [3]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(86517, 11)
(21630, 11)
(27037, 11)


### One-Hot Encoding

In [4]:
blue_champ_cols = [
	"B1Champ", 
	"B2Champ",
	"B3Champ",
	"B4Champ",
	"B5Champ",
]
red_champ_cols = [
	"R1Champ",
	"R2Champ",
	"R3Champ",
	"R4Champ",
	"R5Champ",
]

champ_cols = blue_champ_cols + red_champ_cols
target_cols = ["BlueWin"]

train_unique_champs = get_unique_champs_from_df(train_df, champ_cols)
val_unique_champs = get_unique_champs_from_df(val_df, champ_cols)
test_unique_champs = get_unique_champs_from_df(test_df, champ_cols)

ohe_train = create_one_hot_encoding(
	train_df, 
	blue_champ_cols, 
	red_champ_cols, 
	train_unique_champs, 
	names=True
)

ohe_val = create_one_hot_encoding(
	val_df,
	blue_champ_cols, 
	red_champ_cols, 
	train_unique_champs, 
	names=True
)

ohe_test = create_one_hot_encoding(
	test_df,
	blue_champ_cols,
	red_champ_cols,
	train_unique_champs,
	names=True
)

print(f"Training dataset shape: {ohe_train.shape}")
print("There should be 172 champion cols, 1 target col, and 1 value for each champion (1 for blue team -1 for red")
print(f"Number of cols: {len(ohe_train.columns)}")
print(f"Number of unique cols: {len(ohe_train.columns.unique())}")
print(ohe_train.columns.value_counts())
ohe_train

Training dataset shape: (86517, 173)
There should be 172 champion cols, 1 target col, and 1 value for each champion (1 for blue team -1 for red
Number of cols: 173
Number of unique cols: 173
Annie          1
Olaf           1
Galio          1
TwistedFate    1
Sylas          1
              ..
Bard           1
Naafiri        1
Rakan          1
Xayah          1
BlueWin        1
Name: count, Length: 173, dtype: int64


,Annie,Olaf,Galio,TwistedFate,Sylas,Neeko,Leblanc,XinZhao,Fiddlesticks,Kayle,...,Thresh,Illaoi,RekSai,Ivern,Kalista,Bard,Naafiri,Rakan,Xayah,BlueWin
117776,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,-1,0,0,0,0,1
105311,0,0,0,0,0,-1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41139,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24402,0,0,0,0,-1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
118127,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107132,0,0,0,0,-1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
67393,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
123552,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
59363,-1,0,0,0,0,0,0,0,0,-1,...,0,0,0,0,0,0,0,0,0,1


### Training

In [5]:
X_train = ohe_train.drop(columns=target_cols)
y_train = ohe_train[target_cols].values.ravel()

X_val = ohe_val.drop(columns=target_cols)
y_val = ohe_val[target_cols].values.ravel()

X_test = ohe_test.drop(columns=target_cols)
y_test = ohe_test[target_cols].values.ravel()

In [6]:
# Define hyperparameter grid for tuning
param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
}

best_score = 0
best_params = None
best_model = None

for C in param_grid["C"]:
		model = LogisticRegression(
			C=C,
			max_iter=1000,
			random_state=42,
		)

		# Fit on the training data
		model.fit(X_train, y_train)

		# Evaluate on the validation data. We slice to get the probabilities for the positive class.
		val_prob = model.predict_proba(X_val)[:, 1]
		val_score = roc_auc_score(y_val, val_prob)

		print(f"C={C}, Val AUC={val_score:.4f}")

		if val_score > best_score:
			best_score = val_score
			best_model = model
			best_params = { "C": C }


print(f"\nBest parameters: {best_params}")
print(f"Best validation score: {best_score:.4f}")

# Final evaluation on test set
test_proba = best_model.predict_proba(X_test)[:, 1]
test_score = roc_auc_score(y_test, test_proba)
print(f"Final test score: {test_score:.4f}")

C=0.001, Val AUC=0.5301
C=0.01, Val AUC=0.5309
C=0.1, Val AUC=0.5309
C=1, Val AUC=0.5308
C=10, Val AUC=0.5308
C=100, Val AUC=0.5308

Best parameters: {'C': 0.01}
Best validation score: 0.5309
Final test score: 0.5294
